# Advanced Problems: Randomizing an Iterable Using `sorted`

This notebook explores randomizing iterables with `sorted(..., key=lambda _: random.random())`, while also covering best practices.

Important: sorting by random keys is useful for learning, but it is usually **not** the best production approach. Prefer:

- `random.shuffle()` for shuffling a mutable list in place.
- `random.sample(iterable, k=len(iterable))` for producing a shuffled copy.
- `secrets.SystemRandom()` when randomness must be security-sensitive.

The problems below include full solutions and explanations.

In [1]:
import random
import secrets
from collections import Counter
from pprint import pprint

## Problem 1 — Randomize a list using `sorted`

Use `sorted` with a random key to return a randomized copy of a list. The original list should not be changed.

In [2]:
numbers = list(range(1, 11))
numbers

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

### Solution

In [3]:
randomized = sorted(numbers, key=lambda _: random.random())

print('original:  ', numbers)
print('randomized:', randomized)

original:   [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
randomized: [3, 2, 7, 10, 5, 1, 9, 4, 6, 8]


### Explanation

`sorted` calls the key function once per item. Each item receives a random key, and then the items are sorted by those random keys.

The underscore `_` is used because the actual item value is ignored.

## Problem 2 — Randomize a string and return a string

Randomize the characters of a string using `sorted`, then convert the result back into a string.

In [4]:
text = 'abcdefg'
text

'abcdefg'

### Solution

In [5]:
shuffled_text = ''.join(sorted(text, key=lambda _: random.random()))
shuffled_text

'abegdfc'

### Explanation

Sorting a string returns a list of characters. Use `''.join(...)` to convert the randomized character list back into a string.

## Problem 3 — Write a reusable function for randomizing any iterable

Create a function called `randomized_sorted` that accepts any iterable and returns a randomized list.

The function should support an optional `seed` argument for reproducible results.

### Solution

In [6]:
def randomized_sorted(iterable, seed=None):
    rng = random.Random(seed)
    return sorted(iterable, key=lambda _: rng.random())

print(randomized_sorted(range(10), seed=42))
print(randomized_sorted(range(10), seed=42))

[1, 9, 7, 3, 2, 8, 0, 5, 4, 6]
[1, 9, 7, 3, 2, 8, 0, 5, 4, 6]


### Explanation

`random.Random(seed)` creates a local random number generator. This avoids changing the global random state and makes the function easier to test.

## Problem 4 — Compare `sorted` randomization with `random.shuffle`

Use both approaches to randomize the same list.

Show that:

- `sorted` returns a new list.
- `random.shuffle` mutates a list in place and returns `None`.

### Solution

In [7]:
data = [1, 2, 3, 4, 5]

sorted_version = sorted(data, key=lambda _: random.random())

shuffle_version = data.copy()
shuffle_result = random.shuffle(shuffle_version)

print('original after sorted:', data)
print('sorted version:      ', sorted_version)
print('shuffle version:     ', shuffle_version)
print('shuffle returned:    ', shuffle_result)

original after sorted: [1, 2, 3, 4, 5]
sorted version:       [1, 2, 4, 3, 5]
shuffle version:      [1, 5, 3, 2, 4]
shuffle returned:     None


### Explanation

`random.shuffle()` is usually the best choice when you already have a list and want to shuffle it in place.

`sorted(..., key=random)` is less direct and has `O(n log n)` sorting cost, while shuffling is `O(n)`.

## Problem 5 — Best-practice shuffled copy

Write a function called `shuffled_copy` that accepts any iterable and returns a shuffled list.

Use `random.sample`, not `sorted`.

### Solution

In [8]:
def shuffled_copy(iterable, seed=None):
    items = list(iterable)
    rng = random.Random(seed)
    return rng.sample(items, k=len(items))

print(shuffled_copy('abcdefg', seed=10))
print(''.join(shuffled_copy('abcdefg', seed=10)))

['e', 'a', 'd', 'g', 'c', 'f', 'b']
eadgcfb


### Explanation

`sample(items, k=len(items))` returns a new list containing all items in random order. This is usually clearer than sorting with random keys.

## Problem 6 — Randomize dictionary items

Given a dictionary, return its `(key, value)` pairs in random order.

Use `sorted` with a random key.

In [9]:
scores = {'Ana': 91, 'Ben': 84, 'Cara': 99, 'Diego': 88}
scores

{'Ana': 91, 'Ben': 84, 'Cara': 99, 'Diego': 88}

### Solution

In [10]:
random_items = sorted(scores.items(), key=lambda _: random.random())
random_items

[('Cara', 99), ('Diego', 88), ('Ben', 84), ('Ana', 91)]

### Explanation

Iterating over a dictionary gives keys. To randomize key-value pairs, use `scores.items()`.

## Problem 7 — Reproducible randomized groups

Randomize a list of students and split them into groups of size 3.

The result should be reproducible when a seed is provided.

In [11]:
students = ['Ava', 'Ben', 'Cara', 'Diego', 'Emma', 'Finn', 'Gia', 'Hugo', 'Iris', 'Jade']
students

['Ava', 'Ben', 'Cara', 'Diego', 'Emma', 'Finn', 'Gia', 'Hugo', 'Iris', 'Jade']

### Solution

In [12]:
def make_random_groups(iterable, group_size, seed=None):
    rng = random.Random(seed)
    randomized = rng.sample(list(iterable), k=len(list(iterable)))
    return [randomized[i:i + group_size] for i in range(0, len(randomized), group_size)]

groups = make_random_groups(students, group_size=3, seed=123)
pprint(groups)

[['Ava', 'Emma', 'Ben'],
 ['Gia', 'Diego', 'Cara'],
 ['Jade', 'Finn', 'Hugo'],
 ['Iris']]


### Improved solution

The previous solution works for lists, but it converts the iterable twice. Here is a cleaner version.

In [13]:
def make_random_groups(iterable, group_size, seed=None):
    items = list(iterable)
    rng = random.Random(seed)
    randomized = rng.sample(items, k=len(items))
    return [randomized[i:i + group_size] for i in range(0, len(randomized), group_size)]

groups = make_random_groups(students, group_size=3, seed=123)
pprint(groups)

[['Ava', 'Emma', 'Ben'],
 ['Gia', 'Diego', 'Cara'],
 ['Jade', 'Finn', 'Hugo'],
 ['Iris']]


### Explanation

Always be careful with arbitrary iterables. Some iterables, such as generators, can be consumed only once. Converting to a list once is safer.

## Problem 8 — Demonstrate why random sorting is not ideal

Create a function that randomizes a list using `sorted(..., key=random.random)`. Then compare it conceptually with `random.shuffle`.

Your answer should explain why `shuffle` is preferred.

### Solution

In [14]:
def randomize_with_sorted(iterable):
    return sorted(iterable, key=lambda _: random.random())

def randomize_with_shuffle(iterable):
    items = list(iterable)
    random.shuffle(items)
    return items

data = list(range(10))

print(randomize_with_sorted(data))
print(randomize_with_shuffle(data))

[7, 2, 1, 3, 6, 9, 0, 8, 4, 5]
[9, 8, 1, 5, 2, 7, 3, 0, 4, 6]


### Explanation

`randomize_with_sorted` assigns random keys and sorts by them. This is clever but indirect.

`randomize_with_shuffle` expresses the intent directly: shuffle the items.

Best practice: use sorting when you want ordering; use shuffling when you want random permutation.

## Problem 9 — Count permutation frequencies

For the list `[1, 2, 3]`, repeatedly randomize using `sorted(..., key=lambda _: random.random())` and count how often each permutation occurs.

This is not a formal proof of fairness, but it gives an empirical view.

### Solution

In [15]:
def sorted_random_permutation(items):
    return tuple(sorted(items, key=lambda _: random.random()))

counts = Counter(sorted_random_permutation([1, 2, 3]) for _ in range(20_000))

for permutation, count in sorted(counts.items()):
    print(permutation, count)

(1, 2, 3) 3340
(1, 3, 2) 3303
(2, 1, 3) 3228
(2, 3, 1) 3404
(3, 1, 2) 3414
(3, 2, 1) 3311


### Explanation

There are `3! = 6` possible permutations. With many trials, the counts should be roughly similar.

However, this does not mean sorting by random keys is the best method. It remains less direct and less efficient than `shuffle`.

## Problem 10 — Randomly select and order quiz questions

Given a list of question dictionaries, select 5 questions and return them in random order.

Use best practices, not sorting by random keys.

In [16]:
questions = [
    {'id': 1, 'topic': 'sorting'},
    {'id': 2, 'topic': 'lambdas'},
    {'id': 3, 'topic': 'generators'},
    {'id': 4, 'topic': 'decorators'},
    {'id': 5, 'topic': 'closures'},
    {'id': 6, 'topic': 'iterators'},
    {'id': 7, 'topic': 'recursion'},
    {'id': 8, 'topic': 'classes'},
]
questions

[{'id': 1, 'topic': 'sorting'},
 {'id': 2, 'topic': 'lambdas'},
 {'id': 3, 'topic': 'generators'},
 {'id': 4, 'topic': 'decorators'},
 {'id': 5, 'topic': 'closures'},
 {'id': 6, 'topic': 'iterators'},
 {'id': 7, 'topic': 'recursion'},
 {'id': 8, 'topic': 'classes'}]

### Solution

In [17]:
selected_questions = random.sample(questions, k=5)
pprint(selected_questions)

[{'id': 5, 'topic': 'closures'},
 {'id': 6, 'topic': 'iterators'},
 {'id': 3, 'topic': 'generators'},
 {'id': 7, 'topic': 'recursion'},
 {'id': 8, 'topic': 'classes'}]


### Explanation

`random.sample(questions, k=5)` selects 5 distinct questions and returns them in random order. It is ideal when you want a random subset without replacement.

## Problem 11 — Secure randomization

For games, simulations, and teaching examples, `random` is usually fine.

For security-sensitive use cases, use `secrets` or `secrets.SystemRandom`.

Create a secure shuffled copy of a list.

### Solution

In [18]:
def secure_shuffled_copy(iterable):
    items = list(iterable)
    rng = secrets.SystemRandom()
    rng.shuffle(items)
    return items

secure_shuffled_copy(['A', 'B', 'C', 'D', 'E'])

['C', 'A', 'E', 'D', 'B']

### Explanation

`secrets.SystemRandom()` uses operating-system randomness suitable for security-sensitive tasks. Do not use ordinary `random` for passwords, tokens, secret draws, or anything security-critical.

## Challenge Problem — Randomize while keeping groups balanced

You are given participants with skill levels. Create random groups of size 3, but first sort participants by skill descending, then distribute them round-robin into groups.

Within each skill level, randomize participants.

This combines sorting, randomization, and grouping.

In [19]:
participants = [
    {'name': 'Ava', 'skill': 5},
    {'name': 'Ben', 'skill': 3},
    {'name': 'Cara', 'skill': 5},
    {'name': 'Diego', 'skill': 2},
    {'name': 'Emma', 'skill': 4},
    {'name': 'Finn', 'skill': 3},
    {'name': 'Gia', 'skill': 4},
    {'name': 'Hugo', 'skill': 2},
    {'name': 'Iris', 'skill': 1},
]
participants

[{'name': 'Ava', 'skill': 5},
 {'name': 'Ben', 'skill': 3},
 {'name': 'Cara', 'skill': 5},
 {'name': 'Diego', 'skill': 2},
 {'name': 'Emma', 'skill': 4},
 {'name': 'Finn', 'skill': 3},
 {'name': 'Gia', 'skill': 4},
 {'name': 'Hugo', 'skill': 2},
 {'name': 'Iris', 'skill': 1}]

### Solution

In [20]:
def balanced_random_groups(participants, group_size, seed=None):
    rng = random.Random(seed)

    randomized_within_skill = sorted(
        participants,
        key=lambda p: (-p['skill'], rng.random())
    )

    number_of_groups = math.ceil(len(participants) / group_size)
    groups = [[] for _ in range(number_of_groups)]

    for index, participant in enumerate(randomized_within_skill):
        groups[index % number_of_groups].append(participant)

    return groups

groups = balanced_random_groups(participants, group_size=3, seed=7)
pprint(groups)

NameError: name 'math' is not defined

### Explanation

The key `(-p['skill'], rng.random())` sorts stronger participants first while randomizing ties within the same skill level.

The round-robin distribution then spreads high-skill participants across groups instead of clustering them together.

## Final Best-Practice Summary

Use `sorted(..., key=lambda _: random.random())` when:

- You are learning about key functions.
- You want to demonstrate that keys are computed once per item.
- You are combining deterministic sorting with randomized tie-breaking.

Prefer `random.shuffle()` when:

- You already have a list.
- You want to mutate it in place.

Prefer `random.sample(iterable, k=len(iterable))` when:

- You want a shuffled copy.
- You want to support non-list iterables.

Prefer `secrets.SystemRandom()` when:

- Randomness is security-sensitive.

The main lesson: sorting by random keys is clever, but explicit shuffling is usually clearer, faster, and more idiomatic.